In [2]:
# %%
for key, path in LOGS.items():
    print("=" * 80)
    print(key, "->", path)
    print("=" * 80)
    with open(path, errors="replace") as f:
        lines = f.readlines()
    print("total lines:", len(lines))
    print("---- first 40 lines ----")
    for l in lines[:40]:
        print(repr(l))   # repr 才能看到 ± 之前是不是有空格之类
    # 找第一个 seed 标记后面 20 行
    for i, l in enumerate(lines):
        if "seed=" in l:
            print(f"---- around line {i} (first 'seed=' hit) ----")
            for l2 in lines[i:i+25]:
                print(repr(l2))
            break
    print()

('layernorm', 'relu') -> logs/activation_vari/layernorm_relu_20260918_000114.log


FileNotFoundError: [Errno 2] No such file or directory: 'logs/activation_vari/layernorm_relu_20260918_000114.log'

In [3]:
# %%
import re, json
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

LOG_DIR = Path("logs/activation_vari")
LOGS = {
    ("layernorm", "relu"): LOG_DIR / "layernorm_relu_20260918_000114.log",
    ("layernorm", "gelu"): LOG_DIR / "layernorm_gelu_20260918_005313.log",
    ("none",      "relu"): LOG_DIR / "none_relu_20260918_000114.log",
    ("none",      "gelu"): LOG_DIR / "none_gelu_20260918_005313.log",
}
SEEDS = [42, 43, 44]
SPLITS = ["test_id", "test_cross_device", "test_cross_env", "test_cross_user"]
SPLIT_LABELS = {
    "test_id": "ID",
    "test_cross_device": "Cross-Device",
    "test_cross_env": "Cross-Env",
    "test_cross_user": "Cross-User",
}
SPLIT_COLORS = {
    "test_id": "#64748B",
    "test_cross_device": "#0891B2",
    "test_cross_env": "#7C3AED",
    "test_cross_user": "#D97706",
}

# %%
for key, path in LOGS.items():
    print("=" * 80)
    print(key, "->", path)
    print("=" * 80)
    with open(path, errors="replace") as f:
        lines = f.readlines()
    print("total lines:", len(lines))
    print("---- first 40 lines ----")
    for l in lines[:40]:
        print(repr(l))   # repr 才能看到 ± 之前是不是有空格之类
    # 找第一个 seed 标记后面 20 行
    for i, l in enumerate(lines):
        if "seed=" in l:
            print(f"---- around line {i} (first 'seed=' hit) ----")
            for l2 in lines[i:i+25]:
                print(repr(l2))
            break
    print()
for p in LOGS.values():
    assert p.exists(), f"missing log: {p}"
print("all logs found")

('layernorm', 'relu') -> logs/activation_vari/layernorm_relu_20260918_000114.log


FileNotFoundError: [Errno 2] No such file or directory: 'logs/activation_vari/layernorm_relu_20260918_000114.log'

In [ ]:
# %%
SEED_RE  = re.compile(r"---\s*seed\s*=\s*(\d+)\s*---")
PER_SEED_RE = re.compile(
    r"(?P<split>test_\w+)\s+acc=(?P<acc>[0-9.]+)\s+loss=(?P<loss>[0-9.]+)"
)
AGG_RE = re.compile(
    r"(?P<split>test_\w+)\s+acc=(?P<acc_mean>[0-9.]+)\s*±\s*(?P<acc_std>[0-9.]+)"
    r"\s+f1=(?P<f1_mean>[0-9.]+)\s*±\s*(?P<f1_std>[0-9.]+)"
)

def parse_log(path):
    """Returns:
        per_seed: list of dicts {seed, split, acc, loss}
        agg:      dict {split: {acc_mean, acc_std, f1_mean, f1_std}}
        meta:     dict {norm_type, activation} parsed from header lines (best-effort)
    """
    per_seed = []
    agg = {}
    meta = {}
    current_seed = None

    with open(path, "r", errors="replace") as f:
        for line in f:
            # --- meta ---
            m = re.search(r"norm_type to compare:\s*\[?'?([\w,]+)'?\]?", line)
            if m and "norm_type" not in meta:
                meta["norm_type"] = m.group(1).split(",")[0].strip()
            m = re.search(r"activation:\s*(\w+)", line)
            if m and "activation" not in meta:
                meta["activation"] = m.group(1)

            # --- seed marker ---
            m = SEED_RE.search(line)
            if m:
                current_seed = int(m.group(1))
                continue

            # --- per-seed line: try AGG first (it has ±), then PER_SEED ---
            m = AGG_RE.search(line)
            if m:
                agg[m.group("split")] = {
                    "acc_mean": float(m.group("acc_mean")),
                    "acc_std":  float(m.group("acc_std")),
                    "f1_mean":  float(m.group("f1_mean")),
                    "f1_std":   float(m.group("f1_std")),
                }
                continue

            m = PER_SEED_RE.search(line)
            if m and current_seed is not None and "±" not in line:
                per_seed.append({
                    "seed": current_seed,
                    "split": m.group("split"),
                    "acc": float(m.group("acc")),
                    "loss": float(m.group("loss")),
                })

    return per_seed, agg, meta


parsed = {}
for key, path in LOGS.items():
    per_seed, agg, meta = parse_log(path)
    parsed[key] = {"per_seed": per_seed, "agg": agg, "meta": meta}
    print(f"{key}: {len(per_seed)} per-seed rows, {len(agg)} agg splits  meta={meta}")

In [ ]:
# %%
records = []
for (norm, act), data in parsed.items():
    if data["per_seed"]:
        for r in data["per_seed"]:
            records.append({
                "norm": norm, "activation": act,
                "seed": r["seed"], "split": r["split"],
                "acc": r["acc"], "loss": r["loss"],
            })
    elif data["agg"]:
        for split, v in data["agg"].items():
            records.append({
                "norm": norm, "activation": act,
                "seed": None, "split": split,
                "acc": v["acc_mean"], "loss": None,
            })

df = pd.DataFrame(records)
df["combo"] = df["norm"] + " / " + df["activation"]
print(df.groupby(["norm", "activation", "split"]).size().unstack(fill_value=0))
df.head()

In [ ]:
# %%
def summarize(df):
    g = df.groupby(["norm", "activation", "split"])["acc"]
    out = g.agg(["mean", "std"]).reset_index()
    out["std"] = out["std"].fillna(0.0)
    return out

summ = summarize(df)

# pivot table
table = summ.pivot(index=["norm", "activation"], columns="split", values=["mean", "std"])

pretty = pd.DataFrame(index=table.index)
for split in SPLITS:
    if ("mean", split) in table.columns:
        pretty[SPLIT_LABELS[split]] = (
            table[("mean", split)].round(4).astype(str)
            + " ± " + table[("std", split)].round(4).astype(str)
        )
pretty

In [ ]:
# %%
piv = summ.pivot(index=["norm", "split"], columns="activation", values="mean")
if {"relu", "gelu"}.issubset(piv.columns):
    piv["Δ (gelu − relu)"] = piv["gelu"] - piv["relu"]
    delta = piv.reset_index().pivot(index="norm", columns="split", values="Δ (gelu − relu)")
    delta = delta[[s for s in SPLITS if s in delta.columns]]
    delta.columns = [SPLIT_LABELS[c] for c in delta.columns]
    display(delta.round(4))
else:
    print("calculate only when there is relu and gelu for each norm type")

In [ ]:
# %%
combos = [("layernorm", "relu"), ("layernorm", "gelu"),
          ("none", "relu"),      ("none", "gelu")]
combo_labels = [f"{n}\n{a}" for n, a in combos]

fig, ax = plt.subplots(figsize=(10, 6))
width = 0.2
x = np.arange(len(combos))

for i, split in enumerate(SPLITS):
    means, stds = [], []
    for norm, act in combos:
        row = summ[(summ["norm"] == norm) & (summ["activation"] == act) & (summ["split"] == split)]
        means.append(row["mean"].values[0] if len(row) else np.nan)
        stds.append(row["std"].values[0] if len(row) else 0.0)
    ax.bar(x + (i - 1.5) * width, means, width=width, yerr=stds,
           label=SPLIT_LABELS[split], color=SPLIT_COLORS[split], capsize=3)

ax.set_xticks(x)
ax.set_xticklabels(combo_labels)
ax.set_ylabel("Accuracy")
ax.set_ylim(0, 1.0)
ax.set_title("ReLU vs GELU × LayerNorm vs none  (mean ± std across seeds)")
ax.grid(alpha=0.3, axis="y")
ax.legend(fontsize=9, loc="upper right")
fig.tight_layout()
plt.show()

In [ ]:
# %%
ood_splits = ["test_cross_device", "test_cross_env", "test_cross_user"]
ood = summ[summ["split"].isin(ood_splits)]
ood_avg = ood.groupby(["norm", "activation"])["mean"].mean().reset_index()

fig, ax = plt.subplots(figsize=(6, 5))
xs = np.arange(len(combos))
vals = []
errs = []
for norm, act in combos:
    r = ood_avg[(ood_avg["norm"] == norm) & (ood_avg["activation"] == act)]
    vals.append(r["mean"].values[0] if len(r) else np.nan)
    # 用 per-seed 的 OOD 均值来算 std
    sub = df[(df["norm"] == norm) & (df["activation"] == act) & (df["split"].isin(ood_splits))]
    if sub["seed"].notna().any():
        per_seed = sub.groupby("seed")["acc"].mean()
        errs.append(per_seed.std(ddof=1) if len(per_seed) > 1 else 0.0)
    else:
        errs.append(0.0)

colors = ["#94A3B8", "#64748B", "#FCA5A5", "#EF4444"]
ax.bar(xs, vals, yerr=errs, color=colors, capsize=4)
ax.set_xticks(xs)
ax.set_xticklabels(combo_labels)
ax.set_ylabel("OOD avg accuracy")
ax.set_title("OOD average: ReLU vs GELU, with/without LayerNorm")
ax.grid(alpha=0.3, axis="y")
for xi, v in zip(xs, vals):
    ax.text(xi, v + 0.01, f"{v:.3f}", ha="center", fontsize=9)
fig.tight_layout()
plt.show()

In [ ]:
# %%
fig, axes = plt.subplots(1, len(SPLITS), figsize=(4 * len(SPLITS), 4), sharey=True)
for ax, split in zip(axes, SPLITS):
    sub = df[df["split"] == split]
    for i, (norm, act) in enumerate(combos):
        pts = sub[(sub["norm"] == norm) & (sub["activation"] == act)]["acc"].values
        jitter = np.random.uniform(-0.08, 0.08, size=len(pts))
        ax.scatter(np.full_like(pts, i, dtype=float) + jitter, pts,
                   color=SPLIT_COLORS[split], alpha=0.8, s=40)
        if len(pts):
            ax.hlines(pts.mean(), i - 0.2, i + 0.2, colors="black", linewidth=2)
    ax.set_xticks(range(len(combos)))
    ax.set_xticklabels([f"{n}\n{a}" for n, a in combos], fontsize=8)
    ax.set_title(SPLIT_LABELS[split])
    ax.grid(alpha=0.3, axis="y")
axes[0].set_ylabel("Accuracy")
fig.suptitle("Per-seed accuracies (dots) and mean (bar)")
fig.tight_layout()
plt.show()

In [ ]:
# %%
out_csv = Path("figs/norm_comparison/activation_ablation_summary.csv")
out_csv.parent.mkdir(parents=True, exist_ok=True)
summ.to_csv(out_csv, index=False)
print("saved", out_csv)

# save Δ table if it exists
if "delta" in dir():
    delta.to_csv(out_csv.with_name("activation_ablation_delta.csv"))
    print("saved", out_csv.with_name("activation_ablation_delta.csv"))